# 詩織 SDXL LoRA 学習ノートブック（Kaggle 無料GPU用）

前提: Kaggleの Settings で電話番号認証済み、Notebook の Accelerator = GPU T4 x2、Internet = ON。
データセットは GitHub リポジトリ keen-fox-24/shiori-sd から自動取得する（Kaggle Dataset への手動アップロードは不要）。


In [ ]:
# GPU確認
!nvidia-smi


In [ ]:
# kohya sd-scripts を用意
!git clone https://github.com/kohya-ss/sd-scripts.git /kaggle/working/sd-scripts
%cd /kaggle/working/sd-scripts
!pip install -q -r requirements.txt


## ベースモデル（Animagine XL 4.0）
SDXL系のアニメ向けモデル。Hugging Face から取得する。


In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
import shutil, os
p = hf_hub_download(repo_id='cagliostrolab/animagine-xl-4.0', filename='animagine-xl-4.0.safetensors')
shutil.copy(p, '/kaggle/working/animagine-xl-4.0.safetensors')
print('model ready:', os.path.getsize('/kaggle/working/animagine-xl-4.0.safetensors'))


# データセット配置
GitHub リポジトリ（keen-fox-24/shiori-sd）から zip を取得して展開し、`10_shiori` フォルダ構成に並べる。繰り返し回数10はフォルダ名の接頭辞で指定。


In [ ]:
# データセットを GitHub リポジトリから取得（Kaggle Dataset を経由しない方式）
import os, zipfile, glob, shutil
if not os.path.exists('/kaggle/working/shiori-sd'):
    !git clone https://github.com/keen-fox-24/shiori-sd.git /kaggle/working/shiori-sd
DATASET_ROOT = '/kaggle/working/dataset_flat'
os.makedirs(DATASET_ROOT, exist_ok=True)
with zipfile.ZipFile('/kaggle/working/shiori-sd/shiori-dataset-v2.zip') as z:
    z.extractall(DATASET_ROOT)
print('images:', len(glob.glob(DATASET_ROOT + '/*.png')))
dst = '/kaggle/working/train/10_shiori'
os.makedirs(dst, exist_ok=True)
for f in glob.glob(DATASET_ROOT + '/*.png') + glob.glob(DATASET_ROOT + '/*.txt'):
    shutil.copy(f, dst)
print(sorted(os.listdir(dst)))


## LoRA学習（T4 16GB向け）
1024px / batch 1 / AdamW8bit / fp16 / network_dim 32。T4 x2 のうち1枚を使う想定。
所要時間の目安はデータ枚数とステップ数で変わる。まず小さめ（max_train_epochs=10）で試す。


In [ ]:
%%bash
cd /kaggle/working/sd-scripts
accelerate launch --num_cpu_threads_per_process 4 sdxl_train_network.py \
  --pretrained_model_name_or_path=/kaggle/working/animagine-xl-4.0.safetensors \
  --train_data_dir=/kaggle/working/train \
  --resolution=1024,1024 --enable_bucket --min_bucket_reso=512 --max_bucket_reso=1536 \
  --output_dir=/kaggle/working/output --output_name=shiori_oneesan_v2 \
  --network_module=networks.lora --network_dim=32 --network_alpha=16 \
  --learning_rate=1e-4 --optimizer_type=adafactor --optimizer_args 'relative_step=False' \
  --lr_scheduler=cosine --lr_warmup_steps=100 \
  --max_train_epochs=10 --save_every_n_epochs=2 \
  --mixed_precision=fp16 --cache_latents --cache_latents_to_disk \
  --train_batch_size=1 --gradient_checkpointing --seed=42


## テスト生成（diffusers）
できたLoRAを読み込んで、外出シーンで顔の一致を確認する。


In [ ]:
!pip install -q diffusers transformers accelerate peft
!pip uninstall -y torchao
import torch
from diffusers import StableDiffusionXLPipeline
pipe = StableDiffusionXLPipeline.from_single_file('/kaggle/working/animagine-xl-4.0.safetensors', torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights('/kaggle/working/output', weight_name='shiori_oneesan_v2.safetensors')
prompt = 'shiori_oneesan, 1woman, long black hair, straight hair, bangs, pink eyes, gentle smile, cardigan, shopping street, cafe terrace, sunny day'
neg = 'lowres, bad anatomy, bad hands, extra fingers, worst quality, blurry, multiple girls'
img = pipe(prompt, negative_prompt=neg, width=1024, height=1024, num_inference_steps=28, guidance_scale=7).images[0]
img.save('/kaggle/working/output/test_outing.png')
img
